In [3]:
!git clone https://github.com/devyanirastogi/pedestrian-trajectory-prediction.git

Cloning into 'pedestrian-trajectory-prediction'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 155 (delta 25), reused 141 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 5.62 MiB | 18.62 MiB/s, done.
Resolving deltas: 100% (25/25), done.


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA available: True
GPU: Tesla T4


In [5]:
%pip install ncls


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 85.4 MB/s eta 0:00:00


In [8]:
"""
preprocess_ethucy.py
====================
From-scratch preprocessing of ETH/UCY data into the Environment/Scene/Node
format that Trajectron++ (and therefore MID) expects.

This script does NOT copy-paste from the MID repo. Instead, it rebuilds the
preprocessing logic from first principles, with comments explaining what each
step does and WHY the Trajectron++ encoder needs it.

Dependencies:
    pip install numpy pandas dill

    These are also needed by the environment/ module at import time:
    pip install torch ncls orjson

Overview of what this script produces:
    ┌─────────────────────────────────────────────────────┐
    │ Environment                                         │
    │   - standardization config (mean/std per feature)   │
    │   - attention_radius = 3.0m (who counts as nearby)  │
    │   - node types: just PEDESTRIAN                     │
    │   - scenes: [Scene, Scene, ...]                     │
    │                                                     │
    │   ┌─────────────────────────────────────────────┐   │
    │   │ Scene  (one per .txt file)                   │   │
    │   │   - timesteps: total frames in this file     │   │
    │   │   - dt: 0.4s between frames                  │   │
    │   │   - nodes: [Node, Node, ...]                 │   │
    │   │   - augmented: 24 rotated copies (train)     │   │
    │   │                                              │   │
    │   │   ┌─────────────────────────────────────┐    │   │
    │   │   │ Node  (one per pedestrian)           │    │   │
    │   │   │   - data: (T, 6) array               │    │   │
    │   │   │     cols: pos_x, pos_y,              │    │   │
    │   │   │           vel_x, vel_y,              │    │   │
    │   │   │           acc_x, acc_y               │    │   │
    │   │   │   - first_timestep: when they appear │    │   │
    │   │   │   - type: PEDESTRIAN                 │    │   │
    │   │   └─────────────────────────────────────┘    │   │
    │   └─────────────────────────────────────────────┘   │
    └─────────────────────────────────────────────────────┘

Why this format?
    Trajectron++ (the encoder) expects Scene objects so it can:
    1. Look up which pedestrians are present at each timestep
    2. Compute pairwise distances to build the social interaction graph
       (edges between pedestrians within the attention_radius)
    3. Access position/velocity/acceleration features per node
    4. Standardize features using the Environment's config
"""

import os
import sys
import glob
import numpy as np
import pandas as pd
import dill

# Add project root (parent of notebooks/) to path so `environment` is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from environment import Environment, Scene, Node, derivative_of


# ─────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────
#RAW_DATA_ROOT = "/content/drive/MyDrive/pedestrian-ddpm/raw_data" ##
RAW_DATA_ROOT = os.path.join(PROJECT_ROOT, "raw_data")  # directory containing eth/, hotel/, etc.
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "processed_data")
SCENES = ["eth", "hotel", "univ", "zara1", "zara2"]
SPLITS = ["train", "val", "test"]

FRAME_SKIP = 10    # raw data uses frame IDs 0, 10, 20, ... → normalize to 0, 1, 2, ...
DT = 0.4           # seconds between normalized frames

# Standardization parameters for the Trajectron++ encoder.
# These define how each feature is normalized before being fed to the model.
# Position has std=1 (used as-is), velocity has std=2 (halved), acceleration std=1.
# Mean is 0 for all — we already zero-center positions per scene file.
STANDARDIZATION = {
    "PEDESTRIAN": {
        "position":     {"x": {"mean": 0, "std": 1}, "y": {"mean": 0, "std": 1}},
        "velocity":     {"x": {"mean": 0, "std": 2}, "y": {"mean": 0, "std": 2}},
        "acceleration": {"x": {"mean": 0, "std": 1}, "y": {"mean": 0, "std": 1}},
    }
}

# Social interaction radius: two pedestrians within 3 meters of each other
# get an edge in the scene graph. This is what the GNN encoder uses.
ATTENTION_RADIUS = 3.0

# Training augmentation: rotate trajectories by these angles
AUGMENT_ANGLES = np.arange(0, 360, 15)  # 24 rotations: 0°, 15°, 30°, ..., 345°


# ─────────────────────────────────────────────────────────────────────
# Helper: compute derivatives via finite differences
# ─────────────────────────────────────────────────────────────────────
def compute_velocity(positions, dt):
    """
    Compute velocity from positions using numpy gradient (central differences).

    This is what Trajectron++ expects: not just (x[t] - x[t-1]) / dt,
    but np.gradient which uses central differences for interior points
    and one-sided differences at the boundaries. This gives smoother
    velocity estimates.

    Args:
        positions: 1D array of x or y coordinates
        dt: time step between frames (0.4s)

    Returns:
        velocity: 1D array same length as positions
    """
    if len(positions[~np.isnan(positions)]) < 2:
        return np.zeros_like(positions)

    vel = np.full_like(positions, np.nan)
    valid = ~np.isnan(positions)
    vel[valid] = np.gradient(positions[valid], dt)
    return vel


# ─────────────────────────────────────────────────────────────────────
# Helper: the 6-column DataFrame that Node expects
# ─────────────────────────────────────────────────────────────────────
# Node stores data as a DoubleHeaderNumpyArray with this column structure:
#   (position, x), (position, y), (velocity, x), (velocity, y), (acceleration, x), (acceleration, y)
# We build it as a pandas DataFrame with a MultiIndex on columns,
# then Node.__init__ converts it internally.

DATA_COLUMNS = pd.MultiIndex.from_product(
    [["position", "velocity", "acceleration"], ["x", "y"]]
)


def build_node_dataframe(x_positions, y_positions, dt):
    """
    From raw x,y position arrays, compute all 6 features and return
    the DataFrame that Node expects.

    Features:
        position:     (x, y)  — the raw coordinates
        velocity:     (vx, vy) — first derivative of position
        acceleration: (ax, ay) — second derivative (first derivative of velocity)

    Args:
        x_positions: 1D array of x coordinates for this pedestrian
        y_positions: 1D array of y coordinates
        dt: time step (0.4s)

    Returns:
        pd.DataFrame with 6 columns and MultiIndex headers
    """
    vx = compute_velocity(x_positions, dt)
    vy = compute_velocity(y_positions, dt)
    ax = compute_velocity(vx, dt)  # acceleration = derivative of velocity
    ay = compute_velocity(vy, dt)

    return pd.DataFrame(
        {
            ("position", "x"):     x_positions,
            ("position", "y"):     y_positions,
            ("velocity", "x"):     vx,
            ("velocity", "y"):     vy,
            ("acceleration", "x"): ax,
            ("acceleration", "y"): ay,
        },
        columns=DATA_COLUMNS,
    )


# ─────────────────────────────────────────────────────────────────────
# Helper: rotation augmentation
# ─────────────────────────────────────────────────────────────────────
def rotate_scene(scene, angle_degrees):
    """
    Create a copy of the scene with all trajectories rotated.

    This is a data augmentation trick: since the model should be invariant
    to the absolute orientation of trajectories, we can multiply our
    training data by 24x by rotating everything by 0°, 15°, 30°, ..., 345°.

    Importantly, after rotating positions we RECOMPUTE velocity and
    acceleration from the rotated positions. You can't just rotate the
    velocities because np.gradient depends on the actual coordinate values.
    (Well, mathematically you can rotate velocities too, but recomputing
    is safer and matches what the original code does.)
    """
    angle_rad = angle_degrees * np.pi / 180
    cos_a = np.cos(angle_rad)
    sin_a = np.sin(angle_rad)
    rotation_matrix = np.array([[cos_a, -sin_a],
                                 [sin_a,  cos_a]])

    augmented_scene = Scene(
        timesteps=scene.timesteps,
        dt=scene.dt,
        name=scene.name,
    )

    for node in scene.nodes:
        # Extract original x, y positions
        x_orig = node.data.position.x.copy()
        y_orig = node.data.position.y.copy()

        # Apply 2D rotation: [x', y'] = R @ [x, y]
        coords = np.stack([x_orig, y_orig], axis=0)  # (2, T)
        rotated = rotation_matrix @ coords             # (2, T)
        x_rot, y_rot = rotated[0], rotated[1]

        # Recompute derivatives from rotated positions
        node_df = build_node_dataframe(x_rot, y_rot, scene.dt)

        new_node = Node(
            node_type=node.type,
            node_id=node.id,
            data=node_df,
            first_timestep=node.first_timestep,
        )
        augmented_scene.nodes.append(new_node)

    return augmented_scene


def pick_random_augmentation(scene):
    """Called at training time to randomly select one of the 24 rotations."""
    scene_aug = np.random.choice(scene.augmented)
    scene_aug.temporal_scene_graph = scene.temporal_scene_graph
    return scene_aug


# ─────────────────────────────────────────────────────────────────────
# Core: process one .txt file into a Scene
# ─────────────────────────────────────────────────────────────────────
def txt_to_scene(filepath, scene_name, split, env):
    """
    Read a raw trajectory .txt file and convert it to a Scene object.

    Processing steps:
        1. Parse the 4-column TSV: frame_id, pedestrian_id, x, y
        2. Normalize frame IDs (0, 10, 20, ... → 0, 1, 2, ...)
        3. Apply the ETH test set scaling quirk (0.6x)
        4. Zero-center all positions (subtract global mean of x and y)
        5. For each pedestrian, compute velocity and acceleration
        6. Wrap each pedestrian as a Node, collect into a Scene
        7. For training scenes, pre-compute 24 rotation augmentations

    Args:
        filepath: path to the .txt file
        scene_name: e.g. "eth", "hotel"
        split: "train", "val", or "test"
        env: the Environment object (needed for node type reference)

    Returns:
        Scene object
    """
    # ── Step 1: Parse raw file ──
    # Format: frame_id <tab> pedestrian_id <tab> x <tab> y
    df = pd.read_csv(filepath, sep="\t", header=None)
    df.columns = ["frame", "ped_id", "x", "y"]
    df["frame"] = df["frame"].astype(int)
    df["ped_id"] = df["ped_id"].astype(int)

    # ── Step 2: Normalize frame IDs ──
    # Raw frames are 0, 10, 20, 30, ... (every 10th video frame)
    # We normalize to 0, 1, 2, 3, ... so each step = 0.4 seconds
    df["frame"] = df["frame"] // FRAME_SKIP

    # Shift so the first frame in this file is 0
    df["frame"] -= df["frame"].min()

    df = df.sort_values("frame").reset_index(drop=True)

    # ── Step 3: ETH test set scaling ──
    # The ETH test set (biwi_eth.txt) uses a different coordinate system
    # than the training data. The standard fix is to scale by 0.6.
    # This is a known quirk in the ETH/UCY benchmark.
    if scene_name == "eth" and split == "test":
        df["x"] *= 0.6
        df["y"] *= 0.6

    # ── Step 4: Zero-center positions ──
    # We subtract the mean x and y across ALL pedestrians and ALL frames
    # in this file. This is per-file centering, not per-pedestrian.
    # It roughly centers the scene around the origin.
    df["x"] -= df["x"].mean()
    df["y"] -= df["y"].mean()

    # ── Step 5 & 6: Build Nodes ──
    total_frames = df["frame"].max() + 1

    scene = Scene(
        timesteps=total_frames,
        dt=DT,
        name=f"{scene_name}_{split}",
        aug_func=pick_random_augmentation if split == "train" else None,
    )

    for ped_id, ped_df in df.groupby("ped_id"):
        positions = ped_df[["x", "y"]].values  # (T_ped, 2)

        # Skip pedestrians with fewer than 2 observations
        # (can't compute velocity with just 1 point)
        if positions.shape[0] < 2:
            continue

        first_frame = ped_df["frame"].iloc[0]

        # Build the 6-feature DataFrame
        node_df = build_node_dataframe(
            x_positions=positions[:, 0],
            y_positions=positions[:, 1],
            dt=DT,
        )

        # Create Node
        node = Node(
            node_type=env.NodeType.PEDESTRIAN,
            node_id=str(ped_id),
            data=node_df,
        )
        node.first_timestep = first_frame

        scene.nodes.append(node)

    # ── Step 7: Augmentation (training only) ──
    # Pre-compute 24 rotated versions of the scene.
    # At training time, pick_random_augmentation() selects one at random.
    if split == "train":
        scene.augmented = [
            rotate_scene(scene, angle) for angle in AUGMENT_ANGLES
        ]

    return scene


# ─────────────────────────────────────────────────────────────────────
# Main: process all scenes and splits
# ─────────────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 60)
    print("Preprocessing ETH/UCY → Trajectron++ format")
    print("=" * 60)

    for scene_name in SCENES:
        for split in SPLITS:
            print(f"\n── {scene_name.upper()} / {split} ──")

            # Create a fresh Environment for this split
            env = Environment(
                node_type_list=["PEDESTRIAN"],
                standardization=STANDARDIZATION,
            )
            env.attention_radius = {
                (env.NodeType.PEDESTRIAN, env.NodeType.PEDESTRIAN): ATTENTION_RADIUS,
            }

            # Find all .txt files for this scene/split
            split_dir = os.path.join(RAW_DATA_ROOT, scene_name, split)
            txt_files = sorted(glob.glob(os.path.join(split_dir, "*.txt")))

            if not txt_files:
                print(f"  WARNING: no .txt files in {split_dir}")
                continue

            scenes = []
            for fpath in txt_files:
                fname = os.path.basename(fpath)
                scene = txt_to_scene(fpath, scene_name, split, env)
                print(f"  {fname}: {len(scene.nodes)} pedestrians, "
                      f"{scene.timesteps} frames, "
                      f"{scene.duration():.1f}s duration")
                scenes.append(scene)

            env.scenes = scenes

            # Save as .pkl
            out_path = os.path.join(OUTPUT_DIR, f"{scene_name}_{split}.pkl")
            with open(out_path, "wb") as f:
                dill.dump(env, f, protocol=dill.HIGHEST_PROTOCOL)
            print(f"  → Saved to {out_path}")

    print("\n" + "=" * 60)
    print("Done! Files are in:", OUTPUT_DIR)
    print("=" * 60)
    print("\nTo train MID:")
    print("  python main.py --config configs/baseline.yaml --dataset eth")


if __name__ == "__main__":
    main()

Preprocessing ETH/UCY → Trajectron++ format

── ETH / train ──
  biwi_hotel_train.txt: 310 pedestrians, 1440 frames, 576.0s duration
  crowds_zara01_train.txt: 125 pedestrians, 711 frames, 284.4s duration
  crowds_zara02_train.txt: 171 pedestrians, 841 frames, 336.4s duration
  crowds_zara03_train.txt: 105 pedestrians, 603 frames, 241.2s duration
  students001_train.txt: 373 pedestrians, 355 frames, 142.0s duration
  students003_train.txt: 364 pedestrians, 432 frames, 172.8s duration
  uni_examples_train.txt: 96 pedestrians, 594 frames, 237.6s duration
  → Saved to /content/pedestrian-trajectory-prediction/processed_data/eth_train.pkl

── ETH / val ──
  biwi_hotel_val.txt: 81 pedestrians, 367 frames, 146.8s duration
  crowds_zara01_val.txt: 28 pedestrians, 191 frames, 76.4s duration
  crowds_zara02_val.txt: 47 pedestrians, 211 frames, 84.4s duration
  crowds_zara03_val.txt: 33 pedestrians, 151 frames, 60.4s duration
  students001_val.txt: 95 pedestrians, 89 frames, 35.6s duration
  stu

In [6]:
import os
os.chdir("/content/pedestrian-trajectory-prediction/notebooks")

print(os.getcwd())

/content/pedestrian-trajectory-prediction/notebooks


In [7]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

/content/pedestrian-trajectory-prediction


In [10]:
import json

path = "/content/pedestrian-trajectory-prediction/notebooks/train.ipynb"

with open(path, "r") as f:
    nb = json.load(f)

for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        src = "".join(cell["source"])

        src = src.replace("EPOCHS = 90", "EPOCHS = 20")
        src = src.replace("BATCH_SIZE = 64", "BATCH_SIZE = 128")
        src = src.replace("NUM_DIFFUSION_STEPS = 100", "NUM_DIFFUSION_STEPS = 50")

        cell["source"] = src.splitlines(keepends=True)

with open(path, "w") as f:
    json.dump(nb, f)

print("Updated to 20 epochs.")

Updated to 20 epochs.


In [11]:
with open(path, "r") as f:
    content = f.read()

print("EPOCHS = 20" in content)
print("BATCH_SIZE = 128" in content)
print("NUM_DIFFUSION_STEPS = 50" in content)

True
True
True


In [12]:
%run -i /content/pedestrian-trajectory-prediction/notebooks/train.ipynb

Device: cuda
Loaded /content/pedestrian-trajectory-prediction/processed_data/eth_train.pkl
  scenes: 7
  total nodes: 1544
  attention_radius: {(PEDESTRIAN, PEDESTRIAN): 3.0}

Node type: PEDESTRIAN
Batches per epoch: 296
Encoder constructed with 608,212 parameters
Total params:     7,548,644
Trainable params: 7,548,644


/content/pedestrian-trajectory-prediction/mid_model/diffusion.py:168: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=tf_layer)


Smoke-test loss: 1.0136


Epoch 1/20:   0%|                                                           | 0/296 [00:00<?, ?it/s]

  epoch 1: avg loss = 0.2456  (295.2s)


...

  epoch 18: avg loss = 0.1382  (300.7s)


Epoch 19/20:   0%|                                                          | 0/296 [00:00<?, ?it/s]

  epoch 19: avg loss = 0.1379  (297.6s)


Epoch 20/20:   0%|                                                          | 0/296 [00:00<?, ?it/s]

  epoch 20: avg loss = 0.1360  (299.7s)
Saved checkpoint to ../checkpoints/mid_eth.pt
